# LightSeek-OCR — Training on Colab T4

**Before running:** `Runtime → Change runtime type → T4 GPU`

Checkpoints and the feature cache are saved to Google Drive so they survive session resets.

**Workflow:**
1. Run cells 1–4 (setup)
2. Run cell 5 once to pre-compute encoder features (~1–2h for IAM train, then never again)
3. Run cell 6 to train (~×6 faster than without caching)

In [ ]:
# ── 1. GPU check ────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE            = '/content/drive/MyDrive/lightseek-ocr'
DRIVE_CHECKPOINT_DIR  = f'{DRIVE_BASE}/checkpoints'
DRIVE_METRICS_DIR     = f'{DRIVE_BASE}/metrics'
DRIVE_CACHE_DIR       = f'{DRIVE_BASE}/cache'

import os
for d in [DRIVE_CHECKPOINT_DIR, DRIVE_METRICS_DIR, DRIVE_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Checkpoints → {DRIVE_CHECKPOINT_DIR}')
print(f'Cache       → {DRIVE_CACHE_DIR}')

In [ ]:
# ── 3. Clone repo (branch: feature/refactor) ────────────────────────────────
import os
REPO_DIR = '/content/lightseek-ocr'
BRANCH   = 'feature/refactor'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} https://github.com/Alfred0404/lightseek-ocr.git {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -3

In [ ]:
# ── 4. Install dependencies ──────────────────────────────────────────────────
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers peft accelerate datasets wonderwords jiwer

import torch
print(f'torch {torch.__version__}  |  CUDA {torch.version.cuda}  |  GPU ready: {torch.cuda.is_available()}')

In [ ]:
# ── 5. Pre-compute encoder features (run once, then skip) ───────────────────
#
# SAM + CLIP are frozen → their outputs are the same every epoch.
# This cell computes them once and saves to Drive (~5GB for IAM train).
# On subsequent Colab sessions the cache is already on Drive — skip this cell.
#
# Estimated time: ~1–2h for IAM train split on T4.

import os
REPO_DIR       = '/content/lightseek-ocr'
DRIVE_CACHE_DIR = '/content/drive/MyDrive/lightseek-ocr/cache'

meta_path = os.path.join(DRIVE_CACHE_DIR, 'metadata.json')
if os.path.exists(meta_path):
    import json
    with open(meta_path) as f:
        meta = json.load(f)
    print(f"Cache already exists: {meta['n_samples']} samples — skipping precompute.")
    print("Delete the cache dir on Drive if you want to recompute.")
else:
    print("Starting feature precomputation...")
    !python {REPO_DIR}/scripts/precompute_features.py \
        --dataset iam \
        --split train \
        --cache_dir {DRIVE_CACHE_DIR} \
        --batch_size 8
    print("Done. Cache saved to Drive.")

In [ ]:
# ── 6. Training (cached mode — encoder skipped) ──────────────────────────────
import sys, os
REPO_DIR = '/content/lightseek-ocr'
sys.path.append(f'{REPO_DIR}/src')
sys.path.append(f'{REPO_DIR}/src/train')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/lightseek-ocr/checkpoints'
DRIVE_METRICS_DIR    = '/content/drive/MyDrive/lightseek-ocr/metrics'
DRIVE_CACHE_DIR      = '/content/drive/MyDrive/lightseek-ocr/cache'

COLAB_CONFIG = {
    'DATASET_NAME'      : 'cached',         # use pre-computed features
    'CACHE_DIR'         : DRIVE_CACHE_DIR,
    'BATCH_SIZE'        : 16,               # cached features are small — large batch is fine
    'ACCUMULATION_STEPS': 2,               # effective batch = 32
    'EPOCHS'            : 30,
    'MAX_TEXT_TOKENS'   : 128,
    'CHECKPOINT_DIR'    : DRIVE_CHECKPOINT_DIR,
    'METRICS_DIR'       : DRIVE_METRICS_DIR,
}
print('Config:', COLAB_CONFIG)

In [ ]:
# ── 7. Training loop ────────────────────────────────────────────────────────
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm.notebook import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from LightSeekOCR import LightSeekOCR
from dataset import build_dataset, CachedOCRDataset
from prompt_template import TRANSCRIPTION_PROMPT


def collate_cached(batch):
    local_fs  = torch.stack([b[0] for b in batch])
    global_fs = torch.stack([b[1] for b in batch])
    texts = [b[2] for b in batch]
    return local_fs, global_fs, texts


def plot_loss(epoch_losses, path):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(range(1, len(epoch_losses) + 1), epoch_losses,
            marker='o', color='royalblue', label='Train Loss')
    ax.set(title='Training Loss (live)', xlabel='Epoch', ylabel='Loss')
    ax.legend(); ax.grid(True)
    ax.annotate(f"{epoch_losses[-1]:.4f}",
                xy=(len(epoch_losses), epoch_losses[-1]),
                xytext=(8, 4), textcoords='offset points', fontsize=9, color='royalblue')
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    plt.close(fig)


def train(cfg):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Training on {device} | mode: cached (encoder skipped)')

    model = LightSeekOCR(verbose=True)

    # In cached mode SAM+CLIP are not touched at all — only decoder + visual projection trained
    for param in model.decoder.visual_projection.parameters():
        param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({trainable/total:.2%})')

    lora_params = [p for p in model.decoder.model.parameters() if p.requires_grad]
    optimizer = optim.AdamW([
        {'params': model.decoder.visual_projection.parameters(), 'lr': 1e-4},
        {'params': lora_params,                                  'lr': 5e-5},
    ])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['EPOCHS'])
    scaler    = GradScaler()

    dataset    = build_dataset(name='cached', split='train', cache_dir=cfg['CACHE_DIR'])
    dataloader = DataLoader(dataset, batch_size=cfg['BATCH_SIZE'],
                            shuffle=True, collate_fn=collate_cached, num_workers=2)
    print(f'Dataset: {len(dataset)} samples  |  Batch size: {cfg["BATCH_SIZE"]}')

    tokenizer  = model.decoder.tokenizer
    prompt_ids = tokenizer(TRANSCRIPTION_PROMPT, return_tensors='pt',
                           add_special_tokens=False).input_ids.to(device)
    N_prompt   = prompt_ids.shape[1]
    N_visual   = 512

    os.makedirs(cfg['CHECKPOINT_DIR'], exist_ok=True)
    os.makedirs(cfg['METRICS_DIR'],    exist_ok=True)
    loss_plot_path = os.path.join(cfg['METRICS_DIR'], 'loss_curve.png')

    epoch_losses = []
    step = 0

    for epoch in range(cfg['EPOCHS']):
        model.train()
        epoch_loss, n_samples = 0, 0
        optimizer.zero_grad()

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{cfg['EPOCHS']}", leave=False)
        for local_f, global_f, texts in pbar:
            B = local_f.shape[0]
            local_f  = local_f.to(device).float()   # (B, 256, 768)
            global_f = global_f.to(device).float()  # (B, 256, 768)

            text_inputs = tokenizer(
                [t + tokenizer.eos_token for t in texts],
                return_tensors='pt', padding=True,
                truncation=True, max_length=cfg['MAX_TEXT_TOKENS']
            ).to(device)

            labels_text = text_inputs.input_ids.clone()
            labels_text[text_inputs.attention_mask == 0] = -100

            combined_text_ids  = torch.cat([prompt_ids.expand(B, -1), text_inputs.input_ids], dim=1)
            combined_text_mask = torch.cat([
                torch.ones((B, N_prompt), dtype=torch.long, device=device),
                text_inputs.attention_mask,
            ], dim=1)
            labels = torch.cat([
                torch.full((B, N_visual), -100, dtype=torch.long, device=device),
                torch.full((B, N_prompt), -100, dtype=torch.long, device=device),
                labels_text,
            ], dim=1)

            with autocast(device_type='cuda'):
                outputs = model.decoder(
                    local_features=local_f, global_features=global_f,
                    text_input_ids=combined_text_ids,
                    text_attention_mask=combined_text_mask,
                    labels=labels,
                )

            loss = outputs.loss / cfg['ACCUMULATION_STEPS']
            scaler.scale(loss).backward()
            epoch_loss += outputs.loss.item() * B
            n_samples  += B
            step       += 1

            if step % cfg['ACCUMULATION_STEPS'] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            pbar.set_postfix({'loss': f"{epoch_loss / max(n_samples,1):.4f}"})

        if step % cfg['ACCUMULATION_STEPS'] != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        scheduler.step()
        avg_loss = epoch_loss / max(n_samples, 1)
        epoch_losses.append(avg_loss)
        print(f'Epoch {epoch+1:02d}/{cfg["EPOCHS"]}  Loss: {avg_loss:.4f}')

        ckpt_path = os.path.join(cfg['CHECKPOINT_DIR'], f'model_epoch_{epoch+1}.pth')
        torch.save(model.state_dict(), ckpt_path)
        plot_loss(epoch_losses, loss_plot_path)

    print(f'Done. Loss curve: {loss_plot_path}')


train(COLAB_CONFIG)

In [ ]:
# ── 8. Display loss curve ────────────────────────────────────────────────────
from IPython.display import Image as IPImage
import os
loss_plot = os.path.join('/content/drive/MyDrive/lightseek-ocr/metrics', 'loss_curve.png')
if os.path.exists(loss_plot):
    display(IPImage(loss_plot))
else:
    print('No loss curve yet — run training first.')